<a href="https://colab.research.google.com/github/jstrend/math/blob/main/%ED%81%AC%EB%9E%98%EB%A8%B8%EA%B7%9C%EC%B9%99%EC%9D%84_%EC%9D%B4%EC%9A%A9%ED%95%9C_%EC%84%A0%ED%98%95%EB%B0%A9%EC%A0%95%EC%8B%9D%EC%9D%98_%ED%95%B4p234.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sympy as sp


def input_linear_system():
    """연립일차방정식의 계수행렬 A와 상수항 벡터 b 입력"""

    n = int(input("미지수의 개수를 입력하세요: "))

    print(f"\n{n}×{n} 계수행렬 A를 입력하세요.")
    print("각 행의 계수를 공백으로 구분하여 입력합니다.")
    print("정수, 소수, 분수(예: 1/2)를 입력할 수 있습니다.\n")

    matrix_data = []

    for i in range(n):
        while True:
            values = input(f"계수행렬 {i + 1}행: ").split()

            if len(values) != n:
                print(f"계수를 정확히 {n}개 입력하세요.")
                continue

            try:
                row = [sp.Rational(value) for value in values]
                matrix_data.append(row)
                break

            except ValueError:
                print("올바른 숫자를 입력하세요.")

    print("\n상수항 벡터 b를 입력하세요.")

    while True:
        values = input(f"상수항 {n}개: ").split()

        if len(values) != n:
            print(f"상수항을 정확히 {n}개 입력하세요.")
            continue

        try:
            constant_data = [sp.Rational(value) for value in values]
            break

        except ValueError:
            print("올바른 숫자를 입력하세요.")

    A = sp.Matrix(matrix_data)
    b = sp.Matrix(constant_data)

    return A, b


def cramer_rule(A, b):
    """
    크래머의 규칙으로 연립방정식의 해 계산

    xi = det(Ai) / det(A)
    """

    if A.rows != A.cols:
        raise ValueError("계수행렬 A는 정사각행렬이어야 합니다.")

    if A.rows != b.rows:
        raise ValueError("계수행렬과 상수항 벡터의 크기가 일치하지 않습니다.")

    n = A.cols
    det_A = A.det()

    # det(A)=0이면 크래머의 규칙을 사용할 수 없음
    if det_A == 0:
        return det_A, [], []

    replaced_matrices = []
    solutions = []

    for i in range(n):
        # 계수행렬 복사
        A_i = A.copy()

        # i번째 열을 상수항 벡터 b로 교체
        A_i[:, i] = b

        # 교체된 행렬의 행렬식
        det_A_i = A_i.det()

        # xi = det(Ai) / det(A)
        x_i = sp.simplify(det_A_i / det_A)

        replaced_matrices.append((A_i, det_A_i))
        solutions.append(x_i)

    return det_A, replaced_matrices, solutions


# --------------------------------------------------
# 프로그램 실행
# --------------------------------------------------

A, b = input_linear_system()

det_A, replaced_matrices, solutions = cramer_rule(A, b)

print("\n" + "=" * 50)
print("크래머의 규칙 계산 결과")
print("=" * 50)

print("\n① 계수행렬 A")
sp.pprint(A)

print("\n② 상수항 벡터 b")
sp.pprint(b)

print("\n③ 계수행렬의 행렬식 det(A)")
sp.pprint(det_A)


if det_A == 0:
    print("\ndet(A) = 0이므로 크래머의 규칙을 적용할 수 없습니다.")

    # 해의 존재 여부 확인
    augmented_matrix = A.row_join(b)

    rank_A = A.rank()
    rank_augmented = augmented_matrix.rank()

    print(f"\n계수행렬의 계수 rank(A) = {rank_A}")
    print(f"확대행렬의 계수 rank([A|b]) = {rank_augmented}")

    if rank_A < rank_augmented:
        print("\n이 연립방정식은 해가 없습니다.")
    else:
        print("\n이 연립방정식은 무수히 많은 해를 가집니다.")

else:
    print("\ndet(A) ≠ 0이므로 유일한 해가 존재합니다.")

    print("\n④ 각 열을 상수항 벡터로 교체한 행렬")

    for i, (A_i, det_A_i) in enumerate(replaced_matrices):
        print(f"\nA{i + 1}: A의 {i + 1}번째 열을 b로 교체")
        sp.pprint(A_i)

        print(f"det(A{i + 1}) = {det_A_i}")

        print(
            f"x{i + 1} = det(A{i + 1}) / det(A)"
            f" = {det_A_i} / {det_A}"
            f" = {solutions[i]}"
        )

    print("\n⑤ 연립방정식의 해")

    for i, solution in enumerate(solutions):
        print(f"x{i + 1} = {solution}")

    # 열벡터 형태로 해 출력
    x = sp.Matrix(solutions)

    print("\n해 벡터 x")
    sp.pprint(x)

    # 검산
    print("\n⑥ 검산: A × x")
    verification = sp.simplify(A * x)
    sp.pprint(verification)

    if verification == b:
        print("\n검산 결과: A × x = b이므로 계산된 해가 정확합니다.")
    else:
        print("\n검산 결과를 확인해야 합니다.")